# HRNet
This notebook is used to test the model

## Imports

In [ ]:
import sys
from pathlib import Path
import cv2
import glob
import os
from google.colab.patches import cv2_imshow

In [ ]:
# Set execution root in the project root
sys.path.insert(0, str(Path.cwd().parent))

In [ ]:
from src.utils.config.config import Config
from src.detector.yolo_detector import YOLODetector
from src.pose_estimator.hrnet_pose import HrNetPose
from mmpose.visualization import PoseLocalVisualizer
from mmengine.registry import init_default_scope

## Config

In [ ]:
config_loader = Config()
cfg = config_loader.load_config()

## Dataset

In [ ]:
dataset_path = ""

## Init YOLO

In [ ]:
detector = YOLODetector(cfg.yolo.model_path, cfg.project_root)

## Init HRNet

In [ ]:
pose_estimator = HrNetPose(cfg.hrnet.model_cfg, cfg.hrnet.model_ckpt, cfg.device)

## Test

### Function

In [ ]:
def test_hrnet(detector, 
               pose_estimator, 
               output_path, 
               dataset_path, 
               save_video=True, 
               play_frames=False):
  # Config
  frames_path = sorted(glob.glob(dataset_path + "/test/images/*.jpg"))
  output_video_path = str(output_path / "hrnet_output.mp4")
  
  # Validating the frames path
  if not frames_path:
      raise RuntimeError("No frames found")

  # Ensure the output directory exists
  os.makedirs(output_path, exist_ok=True)

  # Retrieve first frame dimensions
  if save_video:
    first_frame = cv2.imread(frames_path[0])
    height, width = first_frame.shape[:2]
    fps = 5
    # Define the codec and create VideoWriter
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Codec for .mp4 files
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

  init_default_scope('mmpose')
  visualizer = PoseLocalVisualizer()
  visualizer.set_dataset_meta(pose_estimator.model.dataset_meta)

  # Loop through images/frames
  for img_path in frames_path:
    # Read the frame
    frame = cv2.imread(img_path)
    if frame is None:
      continue
    
    # Run through YOLO
    yolo_results = detector.predict(frame)
    if yolo_results is None:
      continue
  
    human_bboxes = []
    for bbox, conf, class_name, class_id in yolo_results:
        if class_name == "person":
            human_bboxes.append(bbox.xyxy.cpu().numpy().squeeze())

    if not human_bboxes:
        continue
    
    # Run HrNet
    pose_results = pose_estimator.infer(frame, human_bboxes)

    print(pose_results)

    vis_frame = visualizer.add_datasample(
        name='result',
        image=frame,
        data_sample=pose_results,
        draw_gt=False,
        draw_bbox=True,
        show=False
    )

    if save_video: out.write(vis_frame) # Write current frame to output
    if play_frames: cv2_imshow(vis_frame) # output current frame

  # Release resources
  if save_video: out.release()

### Run

In [ ]:
test_hrnet(detector, 
           pose_estimator,
           cfg.paths.output_videos, 
           dataset_path,
           save_video=True,
           play_frames=False)